In [ ]:
import yaml
from pathlib import Path

_REPO_ROOT = Path("../..") 
with open(_REPO_ROOT / "configs/data_paths.yaml") as _f:
    _paths = yaml.safe_load(_f)
SAR_ROOT = _paths["SAR_sea_ice_dataset"]


In [ ]:
import numpy as np

In [ ]:
path = f'{SAR_ROOT}/VECTOR_FIELDS_24h_pairs_velocity/HV/region-27_765-81_753-35_0-82_6/2017/04/24/20170424T0638__20170425T0540_future.npz'

In [ ]:
data = np.load(path, allow_pickle=True)

meta = data["meta"].item()
start_path = meta['start_path']
print(start_path)


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

path = start_path

with rasterio.open(path) as ds:
    raw = ds.read(2)

print("RAW:")
print(" NaNs :", np.isnan(raw).sum())
print(" Infs :", np.isinf(raw).sum())
print(" Zeros:", np.sum(raw == 0))

band1 = 10 * np.log10(raw)

print("\nAFTER LOG10:")
print(" NaNs :", np.isnan(band1).sum())
print(" Infs :", np.isinf(band1).sum())

finite = np.isfinite(band1)
mean = band1[finite].mean()
std  = band1[finite].std()

band1_norm = band1#(band1 - mean) / std

band1_norm = np.nan_to_num(band1_norm, nan=0.0, posinf=0.0, neginf=0.0)

plt.imshow(band1_norm, cmap="gray")#, vmin=-4, vmax=4)
plt.colorbar()
plt.title("Band 2 normalized")
plt.show()



In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

path = start_path  # your TIFF

sar_db_eps = 1e-6
apply_db = True

with rasterio.open(path) as ds:
    raw = ds.read(2)  # HV band

print("RAW:")
print(" NaNs :", np.isnan(raw).sum())
print(" Infs :", np.isinf(raw).sum())
print(" Zeros:", np.sum(raw == 0))
print(" Negs :", np.sum(raw < 0))

# --- Step 1: cast to float32
arr = raw.astype(np.float32, copy=False)

# --- Step 2: dB transform (same as dataloader)
if apply_db:
    arr = 10.0 * np.log10(np.maximum(arr, sar_db_eps)).astype(np.float32, copy=False)

print("\nAFTER dB:")
print(" NaNs :", np.isnan(arr).sum())
print(" Infs :", np.isinf(arr).sum())
print(" Min/Max:", np.nanmin(arr), np.nanmax(arr))

# --- Step 3: per-sample normalization (using finite values only)
finite = np.isfinite(arr)

if finite.any():
    mean = arr[finite].mean()
    std = arr[finite].std()
    if std < 1e-6:
        std = 1.0  # avoid division by ~0
else:
    mean = 0.0
    std = 1.0

arr_norm = (arr - mean) / std

print("\nAFTER per-sample normalization:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Mean/std used:", mean, std)

# --- Step 4: force remaining NaNs/Infs to 0
arr_norm = np.nan_to_num(arr_norm, nan=0.0, posinf=0.0, neginf=0.0)

print("\nAFTER NaN/Inf -> 0:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Min/Max:", arr_norm.min(), arr_norm.max())

plt.imshow(arr_norm, cmap="gray")
plt.colorbar()
plt.title("Band after dB + per-sample norm + NaN→0")
plt.show()


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

path = start_path  # your TIFF

sar_db_eps = 1e-6
apply_db = True

with rasterio.open(path) as ds:
    raw = ds.read(2)  # HV band (rasterio is 1-based)

print("RAW:")
print(" NaNs :", np.isnan(raw).sum())
print(" Infs :", np.isinf(raw).sum())
print(" Zeros:", np.sum(raw == 0))
print(" Negs :", np.sum(raw < 0))

# --- Step 1: cast to float32
arr = raw.astype(np.float32, copy=False)

# --- Step 2: dB transform (same as dataloader)
if apply_db:
    arr = 10.0 * np.log10(np.maximum(arr, sar_db_eps)).astype(np.float32, copy=False)

print("\nAFTER dB:")
print(" NaNs :", np.isnan(arr).sum())
print(" Infs :", np.isinf(arr).sum())
print(" Min/Max:", np.nanmin(arr), np.nanmax(arr))

# --- Plot dB distribution (finite values only)
finite_db = np.isfinite(arr)
db_vals = arr[finite_db]

plt.figure(figsize=(6, 4))
plt.hist(db_vals, bins=200, density=True)
plt.xlabel("SAR HV (dB)")
plt.ylabel("Probability density")
plt.title("Distribution of SAR HV values (dB)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Step 3: per-sample normalization (using finite values only)
if finite_db.any():
    mean = db_vals.mean()
    std = db_vals.std()
    if std < 1e-6:
        std = 1.0
else:
    mean = 0.0
    std = 1.0

arr_norm = (arr - mean) / std

print("\nAFTER per-sample normalization:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Mean/std used:", mean, std)

# --- Step 4: force remaining NaNs/Infs to 0
arr_norm = np.nan_to_num(arr_norm, nan=0.0, posinf=0.0, neginf=0.0)

print("\nAFTER NaN/Inf -> 0:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Min/Max:", arr_norm.min(), arr_norm.max())

# --- Plot normalized distribution
finite_norm = np.isfinite(arr_norm)
norm_vals = arr_norm[finite_norm]

plt.figure(figsize=(6, 4))
plt.hist(norm_vals, bins=200, density=True)
plt.xlabel("Normalized SAR HV")
plt.ylabel("Probability density")
plt.title("Distribution after per-sample normalization")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Spatial plot (what the model would see)
plt.figure(figsize=(6, 5))
plt.imshow(arr_norm, cmap="gray")
plt.colorbar(label="Normalized value")
plt.title("Band after dB + per-sample norm + NaN→0")
plt.tight_layout()
plt.show()


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

path = start_path  # your TIFF

sar_db_eps = 1e-6
apply_db = True
p_lo, p_hi = 1.0, 99.0   # percentile clip

with rasterio.open(path) as ds:
    raw = ds.read(1)  # HV band (rasterio is 1-based)

print("RAW:")
print(" NaNs :", np.isnan(raw).sum())
print(" Infs :", np.isinf(raw).sum())
print(" Zeros:", np.sum(raw == 0))
print(" Negs :", np.sum(raw < 0))

# --- Step 1: cast to float32
arr = raw.astype(np.float32, copy=False)

# --- Step 2: dB transform (same as dataloader)
if apply_db:
    arr = 10.0 * np.log10(np.maximum(arr, sar_db_eps)).astype(np.float32, copy=False)

print("\nAFTER dB:")
print(" NaNs :", np.isnan(arr).sum())
print(" Infs :", np.isinf(arr).sum())
print(" Min/Max:", np.nanmin(arr), np.nanmax(arr))

# --- Percentile clipping (finite values only)
finite_db = np.isfinite(arr)
db_vals = arr[finite_db]

if db_vals.size > 0:
    lo = np.percentile(db_vals, p_lo)
    hi = np.percentile(db_vals, p_hi)
else:
    lo, hi = -60.0, 0.0  # safe fallback

arr_clip = np.clip(arr, lo, hi)

print(f"\nAFTER {p_lo:.0f}–{p_hi:.0f} percentile clipping:")
print(" Clip range:", lo, hi)
print(" NaNs :", np.isnan(arr_clip).sum())
print(" Infs :", np.isinf(arr_clip).sum())
print(" Min/Max:", np.nanmin(arr_clip), np.nanmax(arr_clip))

# --- Plot dB distribution (before & after clipping)
plt.figure(figsize=(7, 4))
plt.hist(db_vals, bins=200, density=True, alpha=0.5, label="Original dB")

clip_vals = arr_clip[np.isfinite(arr_clip)]
plt.hist(clip_vals, bins=200, density=True, alpha=0.5, label="Clipped dB")
plt.xlabel("SAR HV (dB)")
plt.ylabel("Probability density")
plt.title("SAR HV distribution before / after percentile clipping")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Step 3: per-sample normalization (using clipped finite values)
finite_clip = np.isfinite(arr_clip)
clip_vals = arr_clip[finite_clip]

if clip_vals.size > 0:
    mean = clip_vals.mean()
    std = clip_vals.std()
    if std < 1e-6:
        std = 1.0
else:
    mean = 0.0
    std = 1.0

arr_norm = (arr_clip - mean) / std

print("\nAFTER per-sample normalization:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Mean/std used:", mean, std)

# --- Step 4: force remaining NaNs/Infs to 0
arr_norm = np.nan_to_num(arr_norm, nan=0.0, posinf=0.0, neginf=0.0)

print("\nAFTER NaN/Inf -> 0:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Min/Max:", arr_norm.min(), arr_norm.max())

# --- Plot normalized distribution
norm_vals = arr_norm[np.isfinite(arr_norm)]

plt.figure(figsize=(6, 4))
plt.hist(norm_vals, bins=200, density=True)
plt.xlabel("Normalized SAR HV")
plt.ylabel("Probability density")
plt.title("Distribution after clipping + normalization")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Spatial plot (what the model would see)
plt.figure(figsize=(6, 5))
plt.imshow(arr_norm, cmap="gray")
plt.colorbar(label="Normalized value")
plt.title("Band after dB + p1–p99 clip + norm + NaN→0")
plt.tight_layout()
plt.show()


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

path = start_path  # your TIFF

sar_db_eps = 1e-6
apply_db = True
p_lo, p_hi = 1.0, 99.0   # percentile clip

with rasterio.open(path) as ds:
    raw = ds.read(1)  # HV band (rasterio is 1-based)

print("RAW:")
print(" dtype:", raw.dtype)
print(" NaNs :", np.isnan(raw).sum())
print(" Infs :", np.isinf(raw).sum())
print(" Zeros:", np.sum(raw == 0))
print(" Negs :", np.sum(raw < 0))

# --- Step 1: cast to float32
arr = raw.astype(np.float32, copy=False)

# NEW: treat ORIGINAL zeros as invalid -> NaN
# (Do this before dB so they don't become -60 dB through eps clamp)
arr[arr == 0] = np.nan

print("\nAFTER zero->NaN (before dB):")
print(" NaNs :", np.isnan(arr).sum())
print(" Infs :", np.isinf(arr).sum())
print(" Zeros:", np.sum(arr == 0))

# --- Step 2: dB transform (similar to dataloader, but NaNs stay NaNs)
if apply_db:
    # preserve NaNs: only transform finite values
    finite_lin = np.isfinite(arr)
    arr_db = np.full_like(arr, np.nan, dtype=np.float32)

    # clamp ONLY finite values to avoid log10(<=0)
    lin = np.maximum(arr[finite_lin], sar_db_eps)
    arr_db[finite_lin] = (10.0 * np.log10(lin)).astype(np.float32, copy=False)

    arr = arr_db

print("\nAFTER dB:")
print(" NaNs :", np.isnan(arr).sum())
print(" Infs :", np.isinf(arr).sum())
print(" Min/Max (finite):", np.nanmin(arr), np.nanmax(arr))

# --- Percentile clipping (finite values only)
finite_db = np.isfinite(arr)
db_vals = arr[finite_db]

if db_vals.size > 0:
    lo = np.percentile(db_vals, p_lo)
    hi = np.percentile(db_vals, p_hi)
else:
    lo, hi = -60.0, 0.0  # safe fallback

arr_clip = arr.copy()
arr_clip[finite_db] = np.clip(arr[finite_db], lo, hi)

print(f"\nAFTER {p_lo:.0f}–{p_hi:.0f} percentile clipping:")
print(" Clip range:", lo, hi)
print(" NaNs :", np.isnan(arr_clip).sum())
print(" Infs :", np.isinf(arr_clip).sum())
print(" Min/Max (finite):", np.nanmin(arr_clip), np.nanmax(arr_clip))

# --- Plot dB distribution (before & after clipping)
plt.figure(figsize=(7, 4))
plt.hist(db_vals, bins=200, density=True, alpha=0.5, label="Original dB")

clip_vals = arr_clip[np.isfinite(arr_clip)]
plt.hist(clip_vals, bins=200, density=True, alpha=0.5, label="Clipped dB")
plt.xlabel("SAR HV (dB)")
plt.ylabel("Probability density")
plt.title("SAR HV distribution before / after percentile clipping")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Step 3: per-sample normalization (using clipped finite values)
finite_clip = np.isfinite(arr_clip)
clip_vals = arr_clip[finite_clip]

if clip_vals.size > 0:
    mean = clip_vals.mean()
    std = clip_vals.std()
    if std < 1e-6:
        std = 1.0
else:
    mean = 0.0
    std = 1.0

arr_norm = (arr_clip - mean) / std

print("\nAFTER per-sample normalization:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Mean/std used:", mean, std)

# --- Step 4: force remaining NaNs/Infs to 0
arr_norm = np.nan_to_num(arr_norm, nan=0.0, posinf=0.0, neginf=0.0)

print("\nAFTER NaN/Inf -> 0:")
print(" NaNs :", np.isnan(arr_norm).sum())
print(" Infs :", np.isinf(arr_norm).sum())
print(" Min/Max:", arr_norm.min(), arr_norm.max())

# --- Plot normalized distribution
norm_vals = arr_norm[np.isfinite(arr_norm)]

plt.figure(figsize=(6, 4))
plt.hist(norm_vals, bins=200, density=True)
plt.xlabel("Normalized SAR HV")
plt.ylabel("Probability density")
plt.title("Distribution after clipping + normalization")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Spatial plot (what the model would see)
plt.figure(figsize=(6, 5))
plt.imshow(arr_norm, cmap="gray")
plt.colorbar(label="Normalized value")
plt.title("Band after dB + p2–p98 clip + norm + NaN→0")
plt.tight_layout()
plt.show()
